# 09b_make_3d_descriptors — 3D descriptor 생성 (신규)

**한 줄 요약:** 1:1 학습셋 분자에 **3차원 입체구조(conformer)** 를 만들어, 모양 기반 **3D descriptor**(구형도·관성모멘트·회전반경 등)를 계산해 저장한다.
**2D descriptor(09)와 차이:** 09는 평면 정보(분자량·logP 등), 여기(09b)는 **입체 모양** 정보. 3D 좌표를 먼저 만들어야 해서 느리다.
**용어:** conformer=3D 좌표를 부여한 분자 모양 / ETKDG=좌표 생성법 / MMFF=에너지 최소화.
**큰 흐름:** ① 준비 → ② 고유 분자 로드 → ③ 3D 생성·descriptor 함수 → ④ 전체 계산·저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
3D 생성(`AllChem`)·3D descriptor(`Descriptors3D`) 도구를 가져온다.

In [ ]:
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors3D
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

🔎 **코드 뜯어보기 (준비)** *(chdir·import는 01에서 설명)*
- `from rdkit.Chem import AllChem, Descriptors3D` : `AllChem`=3D 좌표 생성/최소화, `Descriptors3D`=3D 모양 descriptor 계산.

### 셀 1 — 고유 분자만 추리기
1:1 학습셋에서 **중복 없는 구조**만 뽑는다(같은 구조는 3D도 같아 재계산 낭비 방지).

In [ ]:
# 1:1 학습셋의 '고유 분자'만 (같은 구조는 3D도 같으니 한 번만 계산)
SRC = "data/train_1to1.csv"
OUT = "data/HSD17B13_1to1_3d_descriptors.csv"
df = pd.read_csv(SRC)
smis = df["canonical_smiles"].drop_duplicates().tolist()
print("고유 분자:", len(smis), "개 (3D 생성은 분자당 수십~수백 ms → 수 분 소요)")

🔎 **코드 뜯어보기 (셀 1)**
- `df["canonical_smiles"].drop_duplicates().tolist()` : 중복 제거 후 리스트로. 3D 계산이 비싸서 **고유 구조만** 계산(19에서 다시 붙일 때 이름으로 매칭).

### 셀 2 — 3D 좌표 생성 + 3D descriptor 함수
SMILES→수소 추가→3D 좌표 생성→에너지 최소화→3D descriptor 계산을 한 함수로.

In [ ]:
# SMILES → 3D 좌표(입체구조) 생성 후 3D descriptor 계산
def desc3d(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None or m.GetNumHeavyAtoms() > 60:   # 너무 큰 분자는 3D가 느리고 신뢰도↓ → 건너뜀(NaN)
        return None
    m = Chem.AddHs(m)                       # 3D엔 수소가 필요
    p = AllChem.ETKDGv3(); p.randomSeed = 42; p.maxIterations = 200   # 반복 제한(속도)
    if AllChem.EmbedMolecule(m, p) != 0:    # 3D 좌표 생성 실패 시 바로 포기(속도)
        return None
    # (속도) MMFF 에너지 최소화 생략 — ETKDG 좌표만으로 3D 모양 descriptor 계산에 충분
    return Descriptors3D.CalcMolDescriptors3D(m)   # 3D descriptor 딕셔너리

🔎 **코드 뜯어보기 (셀 2)**
- `Chem.AddHs(m)` : 수소 추가(3D에 필요). `AllChem.ETKDGv3()` : 3D 좌표 생성 설정. `AllChem.EmbedMolecule(m, p)` : **실제 3D 좌표 만들기**(성공하면 0 반환). 실패 시 무작위 좌표로 재시도.
- (속도를 위해 MMFF 에너지 최소화는 생략 — ETKDG 좌표만으로 3D 모양 descriptor 계산에 충분).
- `Descriptors3D.CalcMolDescriptors3D(m)` : 3D 모양 descriptor(구형도·관성모멘트 등)를 **딕셔너리**로 반환.

### 셀 3 — 전체 계산 → 저장
모든 고유 분자에 대해 3D descriptor를 계산해 표로 만들어 CSV로 저장한다(3D 열엔 `d3_` 접두사).

In [ ]:
# 전체 계산 → 표로 저장 (canonical_smiles + 3D descriptor들)
rows, keep = [], []
for i, smi in enumerate(smis):
    d = desc3d(smi)
    if d is None:
        continue
    rows.append(d); keep.append(smi)
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(smis)} (성공 {len(rows)})")

d3 = pd.DataFrame(rows)
d3.columns = ["d3_" + c for c in d3.columns]      # 3D descriptor 열엔 접두사 d3_
d3.insert(0, "canonical_smiles", keep)
d3.to_csv(OUT, index=False)
print(f"\n3D descriptor 저장: {OUT}")
print(f"성공 {len(d3)}/{len(smis)} | 3D descriptor {d3.shape[1]-1}종: "
      + ", ".join([c for c in d3.columns if c != 'canonical_smiles']))

🔎 **코드 뜯어보기 (셀 3)**
- `for i, smi in enumerate(smis):` : 분자를 하나씩(01에서 설명). 실패(None)는 건너뜀.
- `d3.columns = ["d3_" + c for c in d3.columns]` : 열 이름 앞에 **d3_** 를 붙여 2D descriptor와 구분. `.insert(0, ...)`=맨 앞에 SMILES 열 추가. `.to_csv(...)`=저장.